## 1. Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings

warnings.filterwarnings('ignore')

## Configuration 

In [ ]:
class Config:
    PROJECT_NAME = "baseline-rf-features"
    SEED = 42
    
    # Paths
    BASE_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
    TRAIN_DIR = os.path.join(BASE_DIR, "genres_stems")
    TEST_CSV = os.path.join(BASE_DIR, "test.csv")
    SUBMISSION = "submission.csv"
    
    # Audio Params
    SAMPLE_RATE = 22050 
    DURATION = 5.0      # Analyze 5s clips (Sufficient for texture)
    
    # Training
    N_MASHUPS_PER_GENRE = 300 # Create 300 synthetic mashups per genre
    
    GENRES = ["blues", "classical", "country", "disco", "hiphop",
              "jazz", "metal", "pop", "reggae", "rock"]
    GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
    IDX_TO_GENRE = {i: g for g, i in GENRE_TO_IDX.items()}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(Config.SEED)

## Audio processing (Feature Engineering)

In [ ]:
def load_audio(path, duration=Config.DURATION):
    try:
        y, _ = librosa.load(path, sr=Config.SAMPLE_RATE, duration=duration, mono=True)
        # Pad if too short
        target_len = int(Config.SAMPLE_RATE * duration)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        return y[:target_len]
    except:
        return np.zeros(int(Config.SAMPLE_RATE * duration))

def extract_features(y, sr=Config.SAMPLE_RATE):
    """
    Extracts a 1D vector of statistical audio features.
    This is the 'Old School' way that establishes a strong baseline.
    """
    if len(y) == 0: return np.zeros(58) # Fallback size
    
    features = []
    
    # 1. MFCC (Timbre) - Mean & Std
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))
    
    # 2. Spectral Centroid (Brightness)
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    features.extend([np.mean(spec_cent), np.std(spec_cent)])
    
    # 3. Spectral Bandwidth (Texture)
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features.extend([np.mean(spec_bw), np.std(spec_bw)])
    
    # 4. Spectral Rolloff (Frequency Shape)
    spec_roll = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features.extend([np.mean(spec_roll), np.std(spec_roll)])
    
    # 5. Zero Crossing Rate (Percussiveness/Noise)
    zcr = librosa.feature.zero_crossing_rate(y)
    features.extend([np.mean(zcr), np.std(zcr)])
    
    # 6. Chroma (Pitch/Harmony)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features.extend(np.mean(chroma, axis=1))
    
    return np.array(features)

## DATASET GENERATION (MASHUP LOGIC)

In [ ]:
def get_genre_stem_map():
    """Maps genres to all their available stem files."""
    genre_map = {}
    print("🔍 Indexing stems...")
    for genre in Config.GENRES:
        stem_paths = []
        genre_path = os.path.join(Config.TRAIN_DIR, genre)
        if not os.path.exists(genre_path): continue
        
        for root, _, files in os.walk(genre_path):
            for f in files:
                if f.endswith('.wav'):
                    stem_paths.append(os.path.join(root, f))
        genre_map[genre] = stem_paths
    return genre_map

def create_synthetic_dataset():
    """
    Creates a dataset by randomly mixing stems to simulate 'Messy Mashups'.
    """
    genre_map = get_genre_stem_map()
    X, y = [], []
    
    print(f"🧬 Generating {Config.N_MASHUPS_PER_GENRE} mashups per genre...")
    
    for genre, stems in genre_map.items():
        if not stems: continue
        
        for _ in tqdm(range(Config.N_MASHUPS_PER_GENRE), desc=f"{genre}", leave=False):
            # Mix 2-4 random stems to create a mashup
            num_stems_to_mix = random.randint(2, 4)
            chosen_stems = random.sample(stems, min(len(stems), num_stems_to_mix))
            
            # Superimpose stems
            mixed_audio = np.zeros(int(Config.SAMPLE_RATE * Config.DURATION))
            for stem_path in chosen_stems:
                audio = load_audio(stem_path)
                # Random volume for each stem (Robustness)
                vol = random.uniform(0.5, 1.2)
                mixed_audio += audio * vol
            
            # Normalize
            if np.max(np.abs(mixed_audio)) > 0:
                mixed_audio = mixed_audio / np.max(np.abs(mixed_audio))
            
            # Extract Features
            feats = extract_features(mixed_audio)
            X.append(feats)
            y.append(Config.GENRE_TO_IDX[genre])
            
    return np.array(X), np.array(y)

## Main pipeline

In [ ]:
if __name__ == "__main__":
    # --- Step 1: Train ---
    print("\n🚀 Phase 1: Creating Training Data...")
    X_train, y_train = create_synthetic_dataset()
    print(f"✅ Training Data Shape: {X_train.shape}")
    
    print("\n🧠 Phase 2: Training Random Forest...")
    # Random Forest is fast, robust, and handles 1D audio features beautifully
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42))
    ])
    model.fit(X_train, y_train)
    print("✅ Model Trained.")
    
    # --- Step 2: Inference ---
    print("\n🔮 Phase 3: Generating Predictions...")
    test_df = pd.read_csv(Config.TEST_CSV)
    
    # Find test folder (handle potential path variations)
    test_root = Config.BASE_DIR
    sample_filename = test_df.iloc[0]['filename']
    for root, dirs, files in os.walk(Config.BASE_DIR):
        if sample_filename in files:
            test_root = root
            break
            
    X_test = []
    valid_ids = []
    
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Extracting Test Features"):
        file_path = os.path.join(test_root, row['filename'])
        if os.path.exists(file_path):
            audio = load_audio(file_path)
            feats = extract_features(audio)
            X_test.append(feats)
            valid_ids.append(row['id'])
        else:
            # Fallback for missing files (should not happen)
            X_test.append(np.zeros(58))
            valid_ids.append(row['id'])
            
    X_test = np.array(X_test)
    preds = model.predict(X_test)
    
    # --- Step 3: Save ---
    final_genres = [Config.IDX_TO_GENRE[p] for p in preds]
    submission = pd.DataFrame({'id': valid_ids, 'genre': final_genres})
    
    submission.to_csv("submission.csv", index=False)
    print(f"\n✅ Done. Saved 'submission.csv' with {len(submission)} rows.")
    print(submission.head())